<h2>Библиотека Pymorphy</h2>
Руководство пользователя расположено по ссылке:

https://pymorphy2.readthedocs.io/en/stable/user/guide.html


In [ ]:
# Установка
!pip install pymorphy3

# Импорт и инициализация
import pymorphy3

Для морфологического анализа нужно инициализировать экземпляр класса MorphAnalyzer (лучше всего сделать это один раз в начале программы). У него есть метод parse, который возвращает список объектов типа Parse, каждый из которых представляет полный морфологический разбор словоформы.

Морфологический анализ — это определение характеристик слова на основе того, как это слово пишется. При морфологическом анализе не используется информация о соседних словах.

In [ ]:
morph = pymorphy3.MorphAnalyzer()

# разбор слова
word = "стали"
parsed = morph.parse(word)
print(f"Разбор слова '{word}':")
for i, analysis in enumerate(parsed):
    print(f"{i+1}. {analysis}")

https://pymorphy2.readthedocs.io/en/stable/user/grammemes.html

Структура разбора

In [ ]:
word = "бежал"
parsed_word = morph.parse(word)[0]  # Берем наиболее вероятный вариант

print(f"Слово: {parsed_word.word}")
print(f"Нормальная форма: {parsed_word.normal_form}")
print(f"Часть речи: {parsed_word.tag.POS}")
print(f"Падеж: {parsed_word.tag.case}")
print(f"Число: {parsed_word.tag.number}")
print(f"Время: {parsed_word.tag.tense}")
print(f"Полный тег: {parsed_word.tag}")
print(f"Склоняемость: {parsed_word.tag.animacy}")

Работа с множественными разборами

In [ ]:
def analyze_word(word):
    analyses = morph.parse(word)
    print(f"Все варианты разбора слова '{word}':")
    for i, analysis in enumerate(analyses, 1):
        print(f"{i}. НФ: {analysis.normal_form:15} | "
              f"ЧР: {str(analysis.tag.POS):10} | "
              f"Вероятность: {analysis.score:.3f}")

# Тестируем на омонимах
analyze_word("ключ")
analyze_word("печь")
analyze_word("стекло")

Извлечение морфологических признаков

In [ ]:
def get_morph_features(word):
    parsed = morph.parse(word)[0]
    features = {
        'слово': parsed.word,
        'лемма': parsed.normal_form,
        'часть_речи': parsed.tag.POS,
        'падеж': parsed.tag.case,
        'число': parsed.tag.number,
        'род': parsed.tag.gender,
        'время': parsed.tag.tense,
        'залог': parsed.tag.voice,
        'наклонение': parsed.tag.mood
    }
    return {k: v for k, v in features.items() if v is not None}

# Примеры
words = ["столом", "писала", "красивая", "бежали"]
for word in words:
    features = get_morph_features(word)
    print(f"{word}: {features}")

Базовая лемматизация

In [ ]:
def lemmatize_text(text):
    words = text.split()
    lemmas = []

    for word in words:
        # Убираем знаки препинания
        clean_word = ''.join(char for char in word if char.isalpha())
        if clean_word:
            parsed = morph.parse(clean_word)[0]
            lemmas.append(parsed.normal_form)

    return lemmas

text = "Машины ехали по дорогам, обгоняя друг друга"
lemmas = lemmatize_text(text)
print(f"Исходный текст: {text}")
print(f"Леммы: {lemmas}")

Лемматизация с учетом признаков

In [ ]:
def smart_lemmatize(word, pos=None):
    analyses = morph.parse(word)

    if pos:
        # Фильтруем по части речи
        for analysis in analyses:
            if analysis.tag.POS == pos:
                return analysis.normal_form

    # Возвращаем наиболее вероятный вариант
    return analyses[0].normal_form

# Пример с омонимами
print(f"'стекло' как глагол: {smart_lemmatize('стекло', 'VERB')}")
print(f"'стекло' как существительное: {smart_lemmatize('стекло', 'NOUN')}")

Работа с грамматическими тегами в кириллице

In [ ]:
def detailed_analysis(word):
    parsed = morph.parse(word)[0]

    print(f"Детальный разбор слова '{word}':")
    print(f"Лемма: {parsed.normal_form}")
    print(f"Часть речи: {parsed.tag.POS}")
    print(f"Морфологические признаки:")

    # Все доступные признаки
    tag_dict = parsed.tag.cyr_repr  # Кириллическое представление
    print(tag_dict)

detailed_analysis("писавшему")

Фильтрация по грамматическим признакам

In [ ]:
def extract_by_pos(text, target_pos):
    words = text.split()
    result = []

    for word in words:
        clean_word = ''.join(char for char in word if char.isalpha())
        if clean_word:
            parsed = morph.parse(clean_word)[0]
            if parsed.tag.POS == target_pos:
                result.append({
                    'word': clean_word,
                    'lemma': parsed.normal_form,
                    'features': str(parsed.tag)
                })

    return result

text = "Рыжая кошка сидела на окне и смотрела на улицу"

print("Существительные:")
nouns = extract_by_pos(text, 'NOUN')
for noun in nouns:
    print(f"  {noun}")

print("\nГлаголы:")
verbs = extract_by_pos(text, 'VERB')
for verb in verbs:
    print(f"  {verb}")

Поиск слов с определенными характеристиками

In [ ]:
def find_words_with_features(text, **features):
    words = text.split()
    matches = []

    for word in words:
        clean_word = ''.join(char for char in word if char.isalpha())
        if clean_word:
            parsed = morph.parse(clean_word)[0]
            match = True

            for feature, value in features.items():
                if getattr(parsed.tag, feature, None) != value:
                    match = False
                    break

            if match:
                matches.append(parsed.word)

    return matches

text = "Лохматые собаки бежали по узкому тротуару"

# Поиск существительных в именительном падеже
nominative_nouns = find_words_with_features(text, POS='NOUN', case='nomn')
print(f"Существительные в им.падеже: {nominative_nouns}")

Задание 1.
Напишите программу для вычисления лексического разнообразия текста. Оно вычисляется как отношение количества уникальных лексем в слове к количеству словоформ. Вычислите распределение по частям речи.

In [ ]:
text1 = "Как я вскочил на его подножку, было загадкою для меня, в воздухе огненную дорожку он оставлял при свете дня."
text2 = "Мчался он бурей тёмной, крылатой, он заблудился в бездне времён. Остановите, вагоновожатый, остановите сейчас вагон!"

Задание 2.
Проанализировать любой русский текст

1. Найти все глаголы в прошедшем времени
2. Выделить все существительные в дательном падеже
3. Посчитать лексическое разнообразие текста
4. Найти все прилагательные в превосходной степени
5. Создать частотный словарь лемм


Создание морфологического профиля текста

In [ ]:
def create_morph_profile(text):
    """
    Создает морфологический профиль текста, включающий:
    - распределение частей речи
    - разнообразие падежей
    - разнообразие времен глаголов
    - и другие морфологические характеристики
    """
    morph = pymorphy3.MorphAnalyzer()
    words = [word.strip('.,!?;:()""') for word in text.split() if word.strip('.,!?;:()""')]

    profile = {
        'total_words': len(words),
        'pos_distribution': {},
        'case_diversity': {},
        'verb_tense': {},
        'number_distribution': {},
        'gender_distribution': {},
        'unique_lemmas': set(),
        'word_forms_per_lemma': {}
    }

    for word in words:
        parsed = morph.parse(word)[0]  # Берем наиболее вероятный разбор

        # Часть речи
        pos = str(parsed.tag.POS)
        profile['pos_distribution'][pos] = profile['pos_distribution'].get(pos, 0) + 1

        # Падежи (для склоняемых частей речи)
        case = str(parsed.tag.case)
        if case != 'None':
            profile['case_diversity'][case] = profile['case_diversity'].get(case, 0) + 1

        # Время глаголов
        tense = str(parsed.tag.tense)
        if tense != 'None':
            profile['verb_tense'][tense] = profile['verb_tense'].get(tense, 0) + 1

        # Число
        number = str(parsed.tag.number)
        if number != 'None':
            profile['number_distribution'][number] = profile['number_distribution'].get(number, 0) + 1

        # Род
        gender = str(parsed.tag.gender)
        if gender != 'None':
            profile['gender_distribution'][gender] = profile['gender_distribution'].get(gender, 0) + 1

        # Леммы и словоформы
        lemma = parsed.normal_form
        profile['unique_lemmas'].add(lemma)
        if lemma not in profile['word_forms_per_lemma']:
            profile['word_forms_per_lemma'][lemma] = set()
        profile['word_forms_per_lemma'][lemma].add(word)

    return profile

# Тестовый текст
test_text = """
Поздно. Уж обогнули мы стену, мы проскочили сквозь рощу пальм, через Неву, через Нил или Сену мы прогремели по трём мостам.
"""

profile = create_morph_profile(test_text)
print("Морфологический профиль текста:")
for key, value in profile.items():
    if key != 'word_forms_per_lemma':
        print(f"{key}: {value}")

Задание 3. Найдите три текста примерно одинаковой длины: художественный (рассказ или отрывок), научный (статья или учебник), разговорный (диалоги из фильма или интервью).

Проведите анализ для каждого текста: постройте морфологический профиль.

Какой текст самый сложный морфологически и почему?

Какие особенности каждого стиля отражаются в морфологических показателях?

Задание 4.

Возьмите любой текст.
Постройте словари слов: алфавитный список, частотный словарь, частотный словарь лексем.

Постройте алфавитный список всех глаголов. Каждый глагол поставьте в форму первого лица, единственного числа и настоящего времени, если глагол несовершенного вида, и будущего, если он совершенного вида.